In [ ]:
from pathlib import Path
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
import optuna
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import wandb

In [ ]:
p = Path(os.getcwd())
p = p.parents[2]
p1 = p / 'data' / 'processed' / 'simple_prices.parquet'

df = pd.read_parquet(p1)

In [ ]:

clip_matrix = np.vstack(df['v_clip'].values)
brillo = df['brillo'].values.reshape(-1, 1)

X_embeddings = np.hstack([clip_matrix, brillo])
kmeans = KMeans()
clusters = kmeans.fit_predict(X_embeddings)

In [ ]:
pd.Series(clusters).value_counts()

In [ ]:
df['cluster'] = clusters
df

In [ ]:
import pandas as pd

emb = df['v_clip'].apply(pd.Series)
df_final = pd.concat([df.drop(columns=['v_clip']), emb], axis=1)
df = df_final

In [ ]:

lb_price = LabelEncoder()
df['price_range'] = lb_price.fit_transform(df['price_range'])


In [ ]:
y = df['price_range']
X = df.drop('price_range', axis=1)

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=100, random_state=42) 
X_pca = pca.fit_transform(X_scaled)

print("Shape de X original:", X.shape)
print("Shape de X tras PCA:", X_pca.shape)
X = X_pca

In [ ]:
X = pd.DataFrame(X)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

In [ ]:


def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'objective':        'multi:softprob',
        'num_class':        len(np.unique(y_train)),
        'eval_metric':      'mlogloss',
        'tree_method':      'hist',
        'verbosity':        0,
        'random_state':     42,
    }

    num_boost_round = trial.suggest_int('n_estimators', 100, 500)
    early_stopping_rounds = 50

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)

        dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=sample_weights)
        dval = xgb.DMatrix(X_val, label=y_val)

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=num_boost_round,
            evals=[(dval, 'validation')],
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )

        y_pred = np.argmax(model.predict(dval), axis=1)
        score = f1_score(y_val, y_pred, average='weighted')
        scores.append(score)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, n_jobs=1, show_progress_bar=True)

best_params = study.best_params
print(f"\nBest parameters: {best_params}")

In [ ]:
run = wandb.init(
    entity="pd1-c2526-team4",
    project="Precios", 
    name='XGBoost Cluster Clip',
    job_type='xgboost'
)

best_params = {
    'max_depth': 7,
    'n_estimators': 376,
    'learning_rate': 0.038827276088456535,
    'subsample': 0.9913068895561634,
    'colsample_bytree': 0.5107333954752593,
    'reg_alpha': 0.001577941096194598,
    'reg_lambda': 0.00011975580845981367,
    'random_state': 42,
    'tree_method': 'hist',
    'objective': 'multi:softprob',
    'num_class': len(y_train.unique()),
    'eval_metric': 'mlogloss',
    'verbosity': 0,
    'n_jobs': -1,
}
print(best_params)
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = final_model.predict(X_test)

print(classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Confusion Matrix:\n{conf_matrix}")

run.log({
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1-score': f1,
    'confusion-matrix': conf_matrix.tolist() 
    })

run.finish()

In [ ]:
from matplotlib import pyplot as plt

plt.figure(figsize=(12, 8))
xgb.plot_importance(final_model, max_num_features=20, importance_type='weight', height=0.8)
plt.title("Top 20 Features Importancia (XGBoost)")
plt.show()